# M4 Assignment2 - Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL
>PART A: Baseline Model (Frozen Embeddings)

Suchanya Baiyam Business Data Science AAU


* Dataset:AI-Growth-Lab/patents_claims_1.5m_traim_test
* Model: AI-Growth-Lab/PatentSBERTa
* Working file: patents_50k_green.parquet with splits train_silver, pool_unlabeled, eval_silver
* Silver label: is_green_silver (derived from CPC Y02*)

Task
- Train a baseline classifier on train_silver using frozen PatentSBERTa embeddings.
- Evaluate on eval_silver and report Precision / Recall / F1.

In [3]:
!pip install datasets -q
!pip install sentence_transformers -q

STEP01: Import libraries

In [4]:
#from colab patentsberta for patentsearch & suggested by chatgpt for model evaluation task
from datasets import load_dataset
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
#to save the result
import pickle

STEP02: Create balanced 50k dataset
- Download dataset and create the file 'patent_50k_green.parquet'
- Since dataset contains a lot of data, random 500,000 is necessary to cover  25K Green (CPC Y02* Green = 1) / 25K Not Green (No Y02 Green = 0) and make sure it is runable
- Splits: train_silver , pool_unlabeled , eval_silver
- Save as parquet

In [5]:
dataset = load_dataset(
    "AI-Growth-Lab/patents_claims_1.5m_traim_test",
    split="train[:500000]"
)

df = dataset.to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


df_claim_train_1M_pre_duplicates_removed(…):   0%|          | 0.00/3.25G [00:00<?, ?B/s]

df_claim_test_1M_pre_duplicates_removed_(…):   0%|          | 0.00/283M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1372910 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119384 [00:00<?, ? examples/s]

In [6]:
print(df.columns)

Index(['id', 'date', 'text', 'A01B', 'A01C', 'A01D', 'A01F', 'A01G', 'A01H',
       'A01J',
       ...
       'Y02B', 'Y02C', 'Y02D', 'Y02E', 'Y02P', 'Y02T', 'Y02W', 'Y04S', 'Y10S',
       'Y10T'],
      dtype='object', length=666)


In [7]:
#find the columns start with Y02
y02_cols = [col for col in df.columns if col.startswith("Y02")]

#create green label from CPC Y02 / if 1 column of Y02 values 1 = green
df["is_green_silver"] = (df[y02_cols].sum(axis=1) > 0).astype(int)


In [8]:
df["is_green_silver"].value_counts()


,count
is_green_silver,
0,462843
1,37157


In [9]:
df.head()

,id,date,text,A01B,A01C,A01D,A01F,A01G,A01H,A01J,...,Y02C,Y02D,Y02E,Y02P,Y02T,Y02W,Y04S,Y10S,Y10T,is_green_silver
0,8788730,2014-07-22,1. A method for sending a keycode of a non-key...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,8621421,2013-12-31,1. A method executed at least in part in a com...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9461433,2016-10-04,1. A light-emitting device comprising: a base;...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9229528,2016-01-05,"1. An input apparatus, comprising: a plurality...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8508147,2013-08-13,"1. A dimmer circuit, comprising: a bleeder as ...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
#random 25K per class
green_df = df[df["is_green_silver"] == 1].sample(25000, random_state=42)
non_green_df = df[df["is_green_silver"] == 0].sample(25000, random_state=42)
df_balanced = pd.concat([green_df, non_green_df]).sample(frac=1, random_state=42)

#check if 25K/25K or not
print(df_balanced["is_green_silver"].value_counts())

is_green_silver
0    25000
1    25000
Name: count, dtype: int64


In [11]:
#split in ratio of Train 60% with Temporary split 40% including Test 20% Evaluation 20%
#having stratify to make sure it will use equal portion of green and non green everytime
train_df, temp_df = train_test_split(df_balanced, test_size=0.4, random_state=42,stratify=df_balanced["is_green_silver"])
pool_df, eval_df = train_test_split(temp_df, test_size=0.5, random_state=42,stratify=temp_df["is_green_silver"])
train_df["split"] = "train_silver"
pool_df["split"] = "pool_unlabeled"
eval_df["split"] = "eval_silver"
df_final = pd.concat([train_df, pool_df, eval_df])

In [12]:
#store some culumns as instrcuted in part B
df_final = df_final[["id", "text", "is_green_silver", "split"]]
df_final.rename(columns={"id": "doc_id"}, inplace=True)

#save in parquet
df_final.to_parquet("patent_50k_green.parquet")

In [13]:
!ls

drive  patent_50k_green.parquet  sample_data


In [14]:
print(df_final["split"].value_counts())
print(df_final["is_green_silver"].value_counts())
print(df_final.head())

split
train_silver      30000
pool_unlabeled    10000
eval_silver       10000
Name: count, dtype: int64
is_green_silver
0    25000
1    25000
Name: count, dtype: int64
         doc_id                                               text  \
402589  9669654  1. A paint-bucket mesh with a roller cleaning ...   
89394   8984304  1. A method for reducing power consumption in ...   
181025  9234489  1. A method for operating a temperature-limiti...   
25822   9001929  1. A method for a transmitter to transmit data...   
197203  8499387  1. A treatment couch having a lying surface, a...   

        is_green_silver         split  
402589                0  train_silver  
89394                 1  train_silver  
181025                1  train_silver  
25822                 0  train_silver  
197203                0  train_silver  


STEP03: Load working file

In [15]:
df = pd.read_parquet("patent_50k_green.parquet")

In [16]:
train_df = df[df["split"] == "train_silver"]
eval_df = df[df["split"] == "eval_silver"]

In [17]:
print(train_df.shape, eval_df.shape)

(30000, 4) (10000, 4)


STEP04: Load PatentSBERTa (Frozen)
- Model using: AI-Growth-Lab/PatentSBERTa
- no .fit() -> model is freezed

In [18]:
model = SentenceTransformer("AI-Growth-Lab/PatentSBERTa")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

STEP05: Create Embeddings
- Set batch size at 64 will be approximately 469 steps

In [19]:
x_train = model.encode(
    train_df["text"].tolist(),
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=True,
)

x_eval = model.encode(
    eval_df["text"].tolist(),
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=True,
)

y_train = train_df["is_green_silver"].values
y_eval = eval_df["is_green_silver"].values

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [20]:
x_train.shape

(30000, 768)

STEP06: Train logistic regression

In [21]:
clf = LogisticRegression(max_iter=1000)
clf.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

STEP07: Evaluation on eval_silver

In [22]:
y_pred = clf.predict(x_eval)
print(classification_report(y_eval, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.79      0.78      5000
           1       0.78      0.75      0.77      5000

    accuracy                           0.77     10000
   macro avg       0.77      0.77      0.77     10000
weighted avg       0.77      0.77      0.77     10000



STEP08: Save Baseline Model (to work in other colab)

In [23]:
with open("baseline_logreg.pkl", "wb") as f:
    pickle.dump(clf, f)